In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
from pathlib import Path

notebook_path = "/u/skarmakar1/version_check/llm_steering-main/sk"
sys.path.append(notebook_path)

In [3]:
import torch
import numpy as np

from inversion_utils import *
import pickle
from sklearn.model_selection import train_test_split

In [4]:
SEED = 0
# SEED = 1

torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
np.random.seed(SEED)

torch.backends.cudnn.benchmark = True 
torch.backends.cuda.matmul.allow_tf32 = True

LLM = namedtuple('LLM', ['language_model', 'tokenizer', 'processor', 'name', 'model_type'])

In [5]:
model_type = 'llama'
# model_type = 'qwen'

# MODEL_VERSION = '3'
MODEL_VERSION = '3.1'
# MODEL_VERSION = '3.3'

MODEL_SIZE = '8B'
# MODEL_SIZE = '70B'

llm = select_llm(model_type, MODEL_VERSION=MODEL_VERSION, MODEL_SIZE=MODEL_SIZE)

Loading meta-llama/Meta-Llama-3.1-8B-Instruct


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [6]:
hidden_layers = list(range(-1, -llm.language_model.config.num_hidden_layers, -1))
print(hidden_layers)

[-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]


Training

In [7]:
def read_single_translate(llm, fix_lang, other_langs, path, source_constant=True, hidden_layers=None):
    if hidden_layers is None:
        hidden_layers = list(range(-1, -llm.language_model.config.num_hidden_layers, -1))

    Xt = {i: [] for i in hidden_layers}

    if source_constant:
        print("Source Constant")
        for other_lang in other_langs:
            tcontroller = NeuralController(
                llm,
                llm.tokenizer,
                rfm_iters=8,
                control_method="rfm",
                n_components=1
            )

            tcontroller.load_translate(concept=other_lang, model_name=llm.name, orig_lang=fix_lang, path=path)

            dirs = tcontroller.directions

            for k in Xt:
                Xt[k].append(dirs[k])
    else:
        print("Destination Constant")
        for other_lang in other_langs:
            tcontroller = NeuralController(
                llm,
                llm.tokenizer,
                rfm_iters=8,
                control_method="rfm",
                n_components=1
            )

            tcontroller.load_translate(concept=fix_lang, model_name=llm.name, orig_lang=other_lang, path=path)

            dirs = tcontroller.directions

            for k in Xt:
                Xt[k].append(dirs[k])
    
    
    X = {i: torch.cat(Xt[i]).to("cuda") for i in hidden_layers}
    
    return X

In [8]:
# orig_langs = ['english', 'french', 'german', 'hindi', 'italian', 'portuguese', 'spanish', 'thai']

# orig_lang = 'english'
# target_lang = 'italian'
# other_langs = ['french', 'german', 'hindi', 'portuguese', 'spanish', 'thai']

orig_lang = 'french'
target_lang = 'german'
other_langs = ['english', 'italian', 'hindi', 'portuguese', 'spanish', 'thai']


path = '../all_gitignore/language_multi/'

# X = read_single_translate(llm, orig_lang, other_langs, path, hidden_layers=hidden_layers)
# Y = read_single_translate(llm, target_lang, other_langs, path, hidden_layers=hidden_layers)

X = read_single_translate(llm, orig_lang, other_langs, path, source_constant=False, hidden_layers=hidden_layers)
Y = read_single_translate(llm, target_lang, other_langs, path, source_constant=False, hidden_layers=hidden_layers)

Destination Constant
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Detector found
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Detector found
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_comp

In [9]:
print(X[-2].shape)
print(Y[-2].shape)

torch.Size([6, 4096])
torch.Size([6, 4096])


In [10]:
big_X = torch.vstack(list(X.values())).cpu().numpy()
big_Y = torch.vstack(list(Y.values())).cpu().numpy()

In [11]:
print(big_X.shape)
print(big_Y.shape)

(186, 4096)
(186, 4096)


In [12]:
from sklearn.utils import shuffle

big_X_shuffled, big_Y_shuffled = shuffle(big_X, big_Y, random_state=SEED)

In [13]:
model_lrr = LRR_single(big_X_shuffled, big_Y_shuffled, alpha=np.arange(1, 4), print_error=True)

lrr_models = {i: model_lrr for i in hidden_layers}

Running with alpha: [  10.  100. 1000.]
Best lambda: 100.0
Training RMSE: 0.0007, Training R2: 0.9974
Done.


In [14]:
# with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/language/llama8b/source_{orig_lang}TO{target_lang}.pkl', 'wb') as file:
with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/language/llama8b/dest_{orig_lang}TO{target_lang}.pkl', 'wb') as file:
    pickle.dump(lrr_models, file)

Testing

In [19]:
# coef = 0.2
# coef = 0.25
# coef = 0.3
# coef = 0.4
# coef = 0.45
coef = 0.5
# coef = 0.55


max_tokens = 200

orig_lang = 'english'

# prompts = ["What is weight of a football used in fifa?"] # "english"
# prompts = ["फीफा में इस्तेमाल होने वाली फुटबॉल का वज़न कितना होता है?"] # "hindi"
prompts = ["¿Cuál es el peso de un balón de fútbol utilizado en la FIFA?"] # 'spanish'

# l = "german"
# l = "hindi"
# l = "thai"
# l = 'italian'
l = 'portuguese'

l1_controller = load_controller_translate(llm, l, orig_lang, path=f'../all_gitignore/language_multi/')
orig_l = l_controller.directions

l2_controller = load_controller_translate(llm, l, 'spanish', path=f'../all_gitignore/language_multi/')
orig_l = l_controller.directions

out = test_concept_vector(l1_controller, concept=f"coef: {coef}, ques original: {orig_lang}, steered: {l}", prompts=prompts, coef=coef, max_tokens=max_tokens)
out = test_concept_vector(l2_controller, concept=f"coef: {coef}, ques original: {orig_lang}, steered: {l}", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)

Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Detector found
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Detector found

========================== No Control ==========================
¿Cuál es el peso de un balón de fútbol utilizado en la FIFA?
-----------------------------------------------------
Según las reglas de la FIFA, un balón de fútbol oficial debe tener un peso entre 410 y 450 gramos.

========================== + coef: 0.5, que

In [ ]:
# Loading

orig_lang = 'english'
target_lang = 'italian'

with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/language/llama8b/source_{orig_lang}TO{target_lang}.pkl', 'rb') as file:
    lrr_models = pickle.load(file)

In [ ]:
# coef = 0.2
# coef = 0.25
# coef = 0.3
# coef = 0.4
# coef = 0.45
coef = 0.5
# coef = 0.55


max_tokens = 200

prompts = ["What is weight of a football used in fifa?"] # "english"
# prompts = ["फीफा में इस्तेमाल होने वाली फुटबॉल का वज़न कितना होता है?"] # "hindi"
# prompts = ["¿Cuál es el peso de un balón de fútbol utilizado en la FIFA?"] # 'spanish'

# l = "german"
# l = "hindi"
# l = "thai"
# l = 'italian'
l = 'portuguese'

l_controller = load_controller_translate(llm, l, orig_lang, path=f'../all_gitignore/language_multi/')
orig_l = l_controller.directions

out = test_concept_vector(l_controller, concept=f"coef: {coef}, ques original: {orig_lang}, steered: {l}", prompts=prompts, coef=coef, max_tokens=max_tokens)


new_prompts = ["Qual è il peso di un pallone da calcio utilizzato dalla FIFA?"]

out = test_concept_vector(l_controller, concept=f"coef: {coef}, ques original: {target_lang}, steered: {l}", prompts=new_prompts, coef=coef, max_tokens=max_tokens)


trans_l = apply_auto(orig_l, lrr_models)
l_controller.directions = trans_l

out = test_concept_vector(l_controller, concept=f"coef: {coef}, ques original: {target_lang}, M(steered): {l}", prompts=new_prompts, coef=coef, max_tokens=max_tokens, orig=False)

Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Detector found

========================== No Control ==========================
What is weight of a football used in fifa?
-----------------------------------------------------
The weight of a football used in FIFA is 410-450 grams (14.5-15.9 ounces) for the size 5 ball, which is the official size used in professional and international matches.

========================== + coef: 0.5, ques original: english, steered: portuguese Control (normal) ==========================
What is weight of a football used in fifa?
-----------------------------------------------------
O peso de um futebol oficial da FIFA pode variar dependendo do modelo e da marca, mas o peso médio é de

In [15]:
# Loading

# orig_lang = 'english'
# target_lang = 'italian'

orig_lang = 'french'
target_lang = 'german'

with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/language/llama8b/dest_{orig_lang}TO{target_lang}.pkl', 'rb') as file:
    lrr_models = pickle.load(file)

In [19]:
# coef = 0.2
# coef = 0.25
# coef = 0.3
# coef = 0.4
# coef = 0.45
# coef = 0.5
coef = 0.55


max_tokens = 200

# prompts = ["What is weight of a football used in fifa?"] # "english"
# prompts = ["फीफा में इस्तेमाल होने वाली फुटबॉल का वज़न कितना होता है?"] # "hindi"
prompts = ["¿Cuál es el peso de un balón de fútbol utilizado en la FIFA?"] # 'spanish'

# l = "german"
# l = "hindi"
l = "spanish"
# l = "thai"
# l = 'italian'
# l = 'portuguese'

l_controller = load_controller_translate(llm, orig_lang, l, path=f'../all_gitignore/language_multi/')
orig_l = l_controller.directions

out = test_concept_vector(l_controller, concept=f"coef: {coef}, ques original: {l}, steered: {orig_lang}", prompts=prompts, coef=coef, max_tokens=max_tokens)


Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Detector found

========================== No Control ==========================
¿Cuál es el peso de un balón de fútbol utilizado en la FIFA?
-----------------------------------------------------
Según las reglas de la FIFA, un balón de fútbol oficial debe tener un peso entre 410 y 450 gramos.

========================== + coef: 0.55, ques original: spanish, steered: french Control (normal) ==========================
¿Cuál es el peso de un balón de fútbol utilizado en la FIFA?
-----------------------------------------------------
Según las normas de la FIFA, un balón de fútbol utilizado en partidos internationaux doit peser entre 410g et 450g. Cependant, la norme préci

In [24]:
coef = 0.8

trans_l = apply_auto(orig_l, lrr_models)
l_controller.directions = trans_l

out = test_concept_vector(l_controller, concept=f"coef: {coef}, ques original: {l}, M(steered): {target_lang}", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)


========================== + coef: 0.8, ques original: spanish, M(steered): german Control (normal) ==========================
¿Cuál es el peso de un balón de fútbol utilizado en la FIFA?
-----------------------------------------------------
Die Gewichtung eines Fußballs der FIFA ist etwa 450-480 Gramm. Es kann jedoch je nach Hersteller und Material variieren.


Sanity tests

In [12]:
lang1 = "spanish"
lang2 = "english"

target_lang = "hindi"

l1_controller = load_controller_translate(llm, target_lang, lang1, path=f'../all_gitignore/language_multi/')
l2_controller = load_controller_translate(llm, target_lang, lang2, path=f'../all_gitignore/language_multi/')

Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Detector found
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Detector found


In [13]:
compare_cosine(l1_controller.directions, l2_controller.directions)

layer: -1, cosine: 0.6345441341400146
layer: -2, cosine: 0.6440114378929138
layer: -3, cosine: 0.6595911979675293
layer: -4, cosine: 0.6843035221099854
layer: -5, cosine: 0.6993639469146729
layer: -6, cosine: 0.6909059882164001
layer: -7, cosine: 0.7007578611373901
layer: -8, cosine: 0.6582663059234619
layer: -9, cosine: 0.6346402168273926
layer: -10, cosine: 0.6194866299629211
layer: -11, cosine: 0.6064528226852417
layer: -12, cosine: 0.5957133769989014
layer: -13, cosine: 0.24182532727718353
layer: -14, cosine: 0.2508857250213623
layer: -15, cosine: 0.27932631969451904
layer: -16, cosine: 0.5888267755508423
layer: -17, cosine: 0.5975897312164307
layer: -18, cosine: 0.6290510296821594
layer: -19, cosine: 0.640220582485199
layer: -20, cosine: 0.6996731758117676
layer: -21, cosine: 0.7567381262779236
layer: -22, cosine: 0.7832663059234619
layer: -23, cosine: 0.788339376449585
layer: -24, cosine: 0.8005814552307129
layer: -25, cosine: 0.789667010307312
layer: -26, cosine: 0.8032115697860

0.660088257924203

In [15]:
target_lang2 = "italian"

l3_controller = load_controller_translate(llm, target_lang2, lang1, path=f'../all_gitignore/language_multi/')
l4_controller = load_controller_translate(llm, target_lang2, lang2, path=f'../all_gitignore/language_multi/')

Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Detector found
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Detector found


In [18]:
compare_cosine(l4_controller.directions, l2_controller.directions)

layer: -1, cosine: 0.2026747167110443
layer: -2, cosine: 0.2745552062988281
layer: -3, cosine: 0.2866263687610626
layer: -4, cosine: 0.3091799020767212
layer: -5, cosine: 0.3644128441810608
layer: -6, cosine: 0.3524267077445984
layer: -7, cosine: 0.38730746507644653
layer: -8, cosine: 0.39211881160736084
layer: -9, cosine: 0.48528438806533813
layer: -10, cosine: 0.4989360570907593
layer: -11, cosine: 0.5511072278022766
layer: -12, cosine: 0.5706215500831604
layer: -13, cosine: 0.627822756767273
layer: -14, cosine: 0.6990546584129333
layer: -15, cosine: 0.6157951354980469
layer: -16, cosine: 0.6841753125190735
layer: -17, cosine: 0.7062395215034485
layer: -18, cosine: 0.6797906160354614
layer: -19, cosine: 0.7110946178436279
layer: -20, cosine: 0.7553110122680664
layer: -21, cosine: 0.7478247880935669
layer: -22, cosine: 0.7368202209472656
layer: -23, cosine: 0.7257872223854065
layer: -24, cosine: 0.7090833187103271
layer: -25, cosine: 0.7584625482559204
layer: -26, cosine: 0.7063909173

0.5509049892425537